In [1]:
%load_ext autoreload
%autoreload 2
%env ANYWIDGET_HMR=1

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
env: ANYWIDGET_HMR=1


In [5]:
import celldega as dega
import os
dega.__version__

'0.8.1'

In [ ]:
fine_tile_cells['name'] = fine_tile_cells.index.map(cell_str_to_int_mapping)

In [ ]:
df_meta_gene = pd.read_parquet(f"{path_landscape_files}/meta_gene.parquet")
df_meta_gene


,mean,std,max,non-zero,color
ABCC11,0.0,0.637695,2.0,0.000007,#1f77b4
ACE2,0.0,2.751911,7.0,0.000007,#ff7f0e
ACKR1,0.0,3.305460,11.0,0.000007,#2ca02c
ACTA2,0.0,12.725453,24.0,0.000007,#d62728
ACTG2,0.0,15.738799,30.0,0.000007,#9467bd
...,...,...,...,...,...
UnassignedCodeword_0495,0.0,0.042993,1.0,0.000007,#9c9ede
UnassignedCodeword_0496,0.0,0.060802,1.0,0.000007,#6b6ecf
UnassignedCodeword_0497,0.0,0.121604,1.0,0.000007,#5254a3
UnassignedCodeword_0498,0.0,0.000000,0.0,0.000000,#393b79


## Download data if needed

Data source: https://www.10xgenomics.com/datasets/ffpe-human-pancreas-with-xenium-multimodal-cell-segmentation-1-standard

In [4]:
# ! curl -o ../data/Xenium_V1_human_Pancreas_FFPE_outs.zip https://cf.10xgenomics.com/samples/xenium/2.0.0/Xenium_V1_human_Pancreas_FFPE/Xenium_V1_human_Pancreas_FFPE_outs.zip
# ! unzip ../data/Xenium_V1_human_Pancreas_FFPE_outs.zip -d ../data/Xenium_V1_human_Pancreas_FFPE_outs

## Run preprocessing


To run the whole thing in command line
- Git clone celldega repo and 
- cd to celldega/src/celldega, then run:

python src/celldega/pre/run_pre_processing.py \
    --sample Xenium_V1_human_Pancreas_FFPE_outs \
    --data_root_dir data \
    --tile_size 250 \
    --image_tile_layer 'all' \
    --path_landscape_files notebooks/Xenium_V1_human_Pancreas_FFPE_outs


In [6]:
sample = 'Xenium_V1_human_Pancreas_FFPE_outs'

# sample = 'Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_outs'

data_root_dir='../data'
tile_size=250
image_tile_layer='all'
path_landscape_files=f'landscape_files/{sample}'

# dega.pre.main(
#     sample=sample,
#     data_root_dir=data_root_dir,
#     tile_size=tile_size,
#     image_tile_layer=image_tile_layer,
#     path_landscape_files=path_landscape_files,
#     )


In [7]:
technology = 'Xenium'

# Construct data directory
data_dir = os.path.join(data_root_dir, sample)

# Make cell image coordinates
path_transformation_matrix = os.path.join(path_landscape_files, 'micron_to_image_transform.csv')
path_meta_cell_micron = os.path.join(data_dir, 'cells.csv.gz')
path_meta_cell_image = os.path.join(path_landscape_files, 'cell_metadata.parquet')

# Generate transcript tiles
print("\n========Generating transcript tiles========")
path_trx = os.path.join(data_dir, 'transcripts.parquet')
path_trx_tiles = os.path.join(path_landscape_files, 'transcript_tiles')


# tile_bounds = dega.pre.make_trx_tiles(
#     technology,
#     path_trx,
#     path_transformation_matrix,
#     path_trx_tiles,
#     coarse_tile_factor=10,
#     tile_size=tile_size,
#     chunk_size=100000,
#     verbose=False,
#     image_scale=1,
#     max_workers=2
# )
# print (f"tile bounds: {tile_bounds}")

# import pandas as pd
# df = pd.read_parquet(f"{path_trx_tiles}/transcripts_tile_0_49.parquet")
# df.head()



========Generating transcript tiles========


In [8]:
tile_bounds = {'x_min': 0, 'x_max': 34126.65, 'y_min': 0, 'y_max': 13744.4}


# Generate boundary tiles
print("\n========Generating boundary tiles========")
path_cell_boundaries = os.path.join(data_dir, 'cell_boundaries.parquet')
path_output = os.path.join(path_landscape_files, 'cell_segmentation')
cells_orig = dega.pre.make_cell_boundary_tiles(
    technology,
    path_cell_boundaries,
    path_meta_cell_micron,
    path_transformation_matrix,
    path_output,
    coarse_tile_factor=10,
    tile_size=tile_size,
    tile_bounds=tile_bounds,
    image_scale=1,
    max_workers=2
)

import pandas as pd
df = pd.read_parquet(f"{path_output}/cell_tile_0_49.parquet")
df.head()


========Generating boundary tiles========


Processing coarse tiles: 100%|██████████| 14/14 [11:13<00:00, 48.10s/it]


,GEOMETRY,name
0,"[[[247.0, 12469.0], [246.0, 12470.0], [241.0, ...",49751
1,"[[[210.0, 12463.0], [205.0, 12464.0], [201.0, ...",49753
2,"[[[125.0, 12472.0], [123.0, 12474.0], [119.0, ...",49758
3,"[[[172.0, 12466.0], [168.0, 12470.0], [166.0, ...",49759
4,"[[[146.0, 12447.0], [143.0, 12449.0], [135.0, ...",49761


In [ ]:
"""
Pancreas

boundary_tile
- Before: 1m 8s
- After mapping name with integer using  pre-generated dictionary during writing parquet file: 9m 15s
- After mapping name with integer using  pre-generated dictionary after writing parquet file: 1m 20s + 1m 8s (original) = 2m 28s

trx_tile
- Before: 1m
- After mapping name with integer using  pre-generated dictionary during writing parquet file:  <- update
- After mapping name with integer using  pre-generated dictionary after writing parquet file: 3m 52s + 1m (original) = 4m 52s


Ovarian cancer on GCP

After mapping name with integer using  pre-generated dictionary after writing parquet file:
cell_tile: 28min
trx_tile: 21min
"""

In [39]:
# # sample = 'Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_outs'

# def get_directory_size_in_mb(directory):
#     total_size_bytes = 0
#     for dirpath, dirnames, filenames in os.walk(directory):
#         for filename in filenames:
#             filepath = os.path.join(dirpath, filename)
#             total_size_bytes += os.path.getsize(filepath)
#     # Convert bytes to megabytes (1 MB = 1024 * 1024 bytes)
#     total_size_mb = total_size_bytes / (1024 * 1024)
#     return total_size_mb

# # Example usage
# directory_path = f'landscape_files/{sample}/transcript_tiles'
# size_in_mb = get_directory_size_in_mb(directory_path)
# print(f"Total size of '{directory_path}': {size_in_mb:.2f} MB")


## Visualize Landscape files in Celldega

In [9]:
landscape_data_dir = 'landscape_files'
sample = 'Xenium_V1_human_Pancreas_FFPE_outs'

server_address = dega.viz.get_local_server()

landscape_ist = dega.viz.Landscape(
    technology='Xenium',
    base_url = f"http://localhost:{server_address}/{landscape_data_dir}/{sample}",
)

landscape_ist

Server running on port 52809


Landscape(base_url='http://localhost:52809/landscape_files/Xenium_V1_human_Pancreas_FFPE_outs', technology='Xe…

In [ ]:
import glob
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed



def convert_name_col_to_int(tile_fname, str_to_int_mapping):
    df = pd.read_parquet(tile_fname, engine='fastparquet')
    df['name'] = df['name'].map(str_to_int_mapping)
    df.to_parquet(tile_fname, engine='fastparquet', index=False)


gene_str_to_int_mapping, cell_str_to_int_mapping = _get_name_mapping(path_transformation_matrix.replace('/micron_to_image_transform.csv',''))

# List of files to process
files = glob.glob(f"{path_landscape_files}/transcript_tiles/*parquet")

# Process files in parallel
with ThreadPoolExecutor() as executor:
    futures = [executor.submit(convert_name_col_to_int, fname, gene_str_to_int_mapping) for fname in files]
    for future in tqdm(as_completed(futures), total=len(files), desc="Processing files", unit="file"):
        future.result()


# # List of files to process
# files = glob.glob(f"{path_landscape_files}/cell_segmentation/*parquet")

# # Process files in parallel
# with ThreadPoolExecutor() as executor:
#     futures = [executor.submit(process_file, fname, cell_str_to_int_mapping) for fname in files]
#     for future in tqdm(as_completed(futures), total=len(files), desc="Processing files", unit="file"):
#         future.result()

Processing files: 100%|██████████| 7217/7217 [03:47<00:00, 31.70file/s] 


In [ ]:
# # # use original index as name
# # fine_tile_cells = fine_tile_cells.assign(name=fine_tile_cells.index)

# # Use integer name
# fine_tile_cells['name'] = fine_tile_cells.index.map(cell_str_to_int_mapping)

In [7]:
df_meta_cell = pd.read_parquet(f"{path_landscape_files}/cell_metadata.parquet")
df_meta_cell

,name,geometry
cell_id,,
aaaadnje-1,aaaadnje-1,"[2100.3607397615356, 8006.386692719482]"
aaacalai-1,aaacalai-1,"[2076.742577470398, 8168.836703513672]"
aaacjgil-1,aaacjgil-1,"[2193.1913279279174, 8057.692410378906]"
aaacpcil-1,aaacpcil-1,"[2027.5673456346437, 8035.126924938964]"
aaadhocp-1,aaadhocp-1,"[2240.522867346802, 8052.184598291992]"
...,...,...
oiloppgp-1,oiloppgp-1,"[28624.35447082031, 2612.436901953247]"
oilpccne-1,oilpccne-1,"[28738.348028447264, 2329.1849700721436]"
oimacfoj-1,oimacfoj-1,"[28616.427081708985, 2949.374491572632]"


In [11]:
df_meta_cell = df_meta_cell.reset_index(drop=True)
df_meta_cell

,name,geometry
0,aaaadnje-1,"[2100.3607397615356, 8006.386692719482]"
1,aaacalai-1,"[2076.742577470398, 8168.836703513672]"
2,aaacjgil-1,"[2193.1913279279174, 8057.692410378906]"
3,aaacpcil-1,"[2027.5673456346437, 8035.126924938964]"
4,aaadhocp-1,"[2240.522867346802, 8052.184598291992]"
...,...,...
140697,oiloppgp-1,"[28624.35447082031, 2612.436901953247]"
140698,oilpccne-1,"[28738.348028447264, 2329.1849700721436]"
140699,oimacfoj-1,"[28616.427081708985, 2949.374491572632]"
140700,oimaiaae-1,"[28379.26717302539, 2524.7217775576173]"


In [10]:
df_meta_cell['name'].nunique()

140702

In [ ]:
export const set_cell_names_array = (cats, cell_arrow_table) => {
    cats.cell_names_array = cell_arrow_table.getChild("index").toArray();
    // cats.cell_names_array = cell_arrow_table.getChild("name").toArray();

}

export const set_cell_name_to_index_map = (cats) => {
    cats.cell_names_array.forEach((name, index) => {
        cats.cell_name_to_index_map.set(name, index)
    })
}